In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()
        # self.activation = nn.Tanh()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
# X, y = make_friedman1(n_samples=10, noise=1e-2)
# X, y = make_friedman1(n_samples=100, noise=1e-2)
# X, y = make_friedman1(n_samples=1000, noise=1e-2)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(model=model)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0Sparsity: 0.45967250168323515

, loss: 0.1411578208208084
epoch:  1Sparsity: 0.49523188173770905

, loss: 0.12164229154586792
epoch:  2Sparsity: 0.4670681290328503

, loss: 0.10884641855955124
epoch:  3Sparsity: 0.45767500028014185

, loss: 0.08741398900747299
epoch:  4Sparsity: 0.4525587476789951

, loss: 0.0734657272696495
epoch:  5Sparsity: 0.4539749965071678

, loss: 0.061229158192873
epoch:  6Sparsity: 0.5091474995017051

, loss: 0.02494179643690586
epoch:  7Sparsity: 0.5330643713474273

, loss: 0.01606222428381443
epoch:  8Sparsity: 0.5584262490272522

, loss: 0.007018649484962225
epoch:  9Sparsity: 0.5545574977993966

, loss: 0.004878542851656675
epoch:  10Sparsity: 0.5561275005340576

, loss: 0.0036807148717343807
epoch:  11Sparsity: 0.5744656175374985

, loss: 0.0029219260904937983
epoch:  12Sparsity: 0.557730621099472

, loss: 0.002393662929534912
epoch:  13Sparsity: 0.5573593705892563

, loss: 0.0020268098451197147
epoch:  14Sparsity: 0.5587437480688096

, loss: 0.0

KeyboardInterrupt: 

In [ ]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9943689280139562
Test metrics:  R2 = 0.867974852386281


In [ ]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(model=model, batch_size=100)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.15316110849380493
epoch:  1, loss: 0.1490262895822525
epoch:  2, loss: 0.12185228615999222
epoch:  3, loss: 0.09953359514474869
epoch:  4, loss: 0.08521903306245804
epoch:  5, loss: 0.07280026376247406
epoch:  6, loss: 0.030662216246128082
epoch:  7, loss: 0.018040768802165985
epoch:  8, loss: 0.008037683553993702
epoch:  9, loss: 0.005843249149620533
epoch:  10, loss: 0.004327207338064909
epoch:  11, loss: 0.003307812148705125
epoch:  12, loss: 0.0025971161667257547
epoch:  13, loss: 0.0021633710712194443
epoch:  14, loss: 0.001844762242399156
epoch:  15, loss: 0.0016374222468584776
epoch:  16, loss: 0.001479297294281423
epoch:  17, loss: 0.0013608289882540703
epoch:  18, loss: 0.0012670081341639161
epoch:  19, loss: 0.0011852191528305411
epoch:  20, loss: 0.0011086093727499247
epoch:  21, loss: 0.001046984689310193
epoch:  22, loss: 0.0009967449586838484
epoch:  23, loss: 0.0009497951250523329
epoch:  24, loss: 0.0009069595253095031
epoch:  25, loss: 0.000871962751

In [ ]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9961423693356614
Test metrics:  R2 = 0.8418329552900294
